In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import zuko

In [ ]:
def target_pdf(y):
    m_Z = 91.1876     
    Gamma_Z = 2.4952  
    
    # STEP 1: THE BOUNDING BOX (y -> Physical Variables)
    sig_y0 = torch.sigmoid(y[:, 0])
    sig_y1 = torch.sigmoid(y[:, 1])
    
    E = 50.0 + 100.0 * sig_y0
    cos_theta = -1.0 + 2.0 * sig_y1
    
    # STEP 2: TRACK THE SQUISH (HACK-PROOF JACOBIAN)
    log_jac_0 = math.log(100.0) + F.logsigmoid(y[:, 0]) + F.logsigmoid(-y[:, 0])
    log_jac_1 = math.log(2.0) + F.logsigmoid(y[:, 1]) + F.logsigmoid(-y[:, 1])
    log_bounding_jac = log_jac_0 + log_jac_1
    
    # STEP 3: THE QUANTUM MECHANICS 
    s = E**2
    bw_numerator = s
    bw_denominator = (s - m_Z**2)**2 + (m_Z * Gamma_Z)**2
    breit_wigner = bw_numerator / bw_denominator
    
    angular = 1.0 + cos_theta**2
    f_phys = breit_wigner * angular
    
    # STEP 4: FINAL COMBINATION
    log_f_y = torch.log(f_phys) + log_bounding_jac
    return log_f_y

In [ ]:
def train_flow():
    dim = 2
    
    # --- THE NEURAL SPLINE FLOW ARCHITECTURE ---
    # features=2: We are in 2D space
    # transforms=6: 6 Sequential Spline layers (The goldilocks zone)
    # bins=8: The polynomial curve is made of 8 smooth pieces per dimension
    # hidden_features=[64, 64]: The internal MLP is 2 layers deep, 64 neurons wide
    flow = zuko.flows.NSF(features=dim, context=0, transforms=6, bins=8, hidden_features=[64, 64])
    
    optimizer = optim.Adam(flow.parameters(), lr=2e-3)
    epochs = 3000 
    batch_size = 8192 
    
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    beta_start = 0.05       
    beta_end = 1.0          
    anneal_epochs = 2000
    
    best_loss = float('inf')
    patience_counter = 0
    patience_limit = 300  
    
    print("Starting Training with Zuko Neural Spline Flow...")
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        beta = beta_start + (beta_end - beta_start) * min(1.0, epoch / anneal_epochs)
            
        z = torch.randn(batch_size, dim)
        log_p_z = -0.5 * (z**2).sum(dim=-1) - (dim / 2.0) * math.log(2 * math.pi)
        
        # --- THE ZUKO TRANSFORM ---
        # We extract the pure mathematical mapping from the Zuko distribution
        transform = flow().transform
        
        # We push noise 'z' through the spline to get coordinates 'x'
        x = transform(z)
        
        # We ask Zuko for the exact Log-Jacobian of the spline stretch
        log_det = transform.log_abs_det_jacobian(z, x)
        
        log_q_x = log_p_z - log_det
        log_f_x = target_pdf(x)
        
        loss = -(beta * log_f_x - log_q_x).mean()
        loss.backward()
        
        nn.utils.clip_grad_norm_(flow.parameters(), max_norm=1.0)
        
        optimizer.step()
        scheduler.step() 
        
        current_loss = loss.item()
        current_lr = scheduler.get_last_lr()[0]
        
        if epoch % 400 == 0:
            print(f"Epoch {epoch:4d} | Beta: {beta:.3f} | LR: {current_lr:.5f} | Loss: {current_loss:.4f}")
            
        # =========================================================
        # THE EARLY STOPPING MONITOR
        # =========================================================
        if epoch > anneal_epochs:
            if current_loss < best_loss - 1e-4:
                best_loss = current_loss
                patience_counter = 0  
            else:
                patience_counter += 1 
                
            if patience_counter >= patience_limit:
                print(f"\n[EARLY STOPPING TRIGGERED]")
                print(f"Loss completely flatlined for {patience_limit} epochs.")
                print(f"Training successfully terminated early at Epoch {epoch} to save time.")
                break 

    return flow

In [ ]:
# ==========================================
# INTEGRATION AND PLOTTING
# ==========================================
def integrate_and_plot(flow, num_samples=5000000): # Dropped to 5 Million to prevent RAM crash!
    print("\nStarting Physics Integration...")
    dim = 2
    
    # We must explicitly tell Zuko we are evaluating, not training
    flow.eval()
    
    with torch.no_grad():
        z = torch.randn(num_samples, 2)
        log_p_z = -0.5 * (z**2).sum(dim=-1) - (dim / 2.0) * math.log(2 * math.pi)
        
        # Map the testing points through the trained Zuko Splines
        transform = flow().transform
        y = transform(z)
        log_det = transform.log_abs_det_jacobian(z, y)
        
        log_q_y = log_p_z - log_det
        log_f_y = target_pdf(y)
        
        weights = torch.exp(log_f_y - log_q_y)
        
        integral = weights.mean().item()
        variance = weights.var().item()
        error = math.sqrt(variance / num_samples)
        
        print(f"Calculated Integral : {integral:.6f} +/- {error:.6f}")
        print(f"Variance (sigma^2)  : {variance:.6f}")
        
        # --- CONVERT TO PHYSICAL KINEMATICS ---
        sig_y0 = torch.sigmoid(y[:, 0])
        sig_y1 = torch.sigmoid(y[:, 1])
        E = 50.0 + 100.0 * sig_y0
        cos_theta = -1.0 + 2.0 * sig_y1
        
        E_np = E.numpy()
        cos_np = cos_theta.numpy()
    
    m_Z = 91.1876
    plt.figure(figsize=(9, 6))
    plt.hist2d(E_np, cos_np, bins=120, cmap='magma', range=[[80, 105], [-1, 1]])
    plt.colorbar(label='Generated Event Density')
    
    plt.axvline(m_Z, color='white', linestyle='--', alpha=0.7, label=f'Z Mass ({m_Z} GeV)')
    
    plt.title("Learned Z Boson Resonance (Neural Spline Flow)")
    plt.xlabel("Invariant Mass $E$ [GeV]")
    plt.ylabel("Scattering Angle $\\cos(\\theta)$")
    plt.legend()
    plt.show()

In [ ]:
trained_model = train_flow()

In [ ]:
integrate_and_plot(trained_model)